# Notebook 01 - Data Loading and Initial Exploration

## Dataset: UCI Online Retail

This notebook loads the **Online Retail dataset** from the UCI Machine Learning Repository. The dataset contains all transactions for a UK-based online gift retailer between **December 2010 and December 2011**.

### Why this dataset?
It is a real-world, transaction-level dataset containing customer purchases, product descriptions, quantities, prices, and country information - making it ideal for customer analytics, segmentation, and recommendation tasks.

### Columns
| Column | Description |
|--------|-------------|
| InvoiceNo | Unique invoice number. Prefix 'C' indicates a cancellation |
| StockCode | Product code |
| Description | Product name |
| Quantity | Number of units per transaction |
| InvoiceDate | Date and time of the transaction |
| UnitPrice | Price per unit in GBP |
| CustomerID | Unique customer identifier (nullable) |
| Country | Country where the customer resides |

In [2]:
import pandas as pd
import numpy as np

## Step 1: Load the Raw Data

We load the Excel file directly using pandas. This preserves date formatting and data types.

In [4]:
df = pd.read_excel("Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Step 2: Dataset Shape

The shape tells us the total number of transactions (rows) and features (columns).

In [6]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 541,909
Columns: 8


## Step 3: Data Types and Missing Values

Understanding data types helps us plan cleaning steps. Key observations:
- **CustomerID** has ~135,000 missing values - transactions without a customer ID cannot be used for customer-level analysis
- **Description** has ~1,454 missing values - minor, can be dropped during cleaning
- **InvoiceDate** is already parsed as datetime

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


## Step 4: Missing Value Summary

In [10]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

             Missing Count  Missing %
Description           1454       0.27
CustomerID          135080      24.93


## Step 5: Statistical Summary

Key observations from the descriptive statistics:
- **Quantity** has negative values - these are returns/cancellations (will be removed in cleaning)
- **UnitPrice** also has negative values - invalid/erroneous entries (will be removed)
- **CustomerID** ranges from ~12,346 to ~18,287 - about 4,300 unique customers after cleaning

In [12]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## Step 6: Country Distribution

The dataset is heavily skewed toward the **United Kingdom** (~91% of transactions). The remaining ~9% covers international customers across Europe, Australia, and beyond. This geographic bias is important to note when interpreting customer-level analysis.

In [14]:
country_counts = df['Country'].value_counts().head(10)
country_pct = (country_counts / len(df) * 100).round(2)
print(pd.DataFrame({'Count': country_counts, 'Percentage %': country_pct}))

                 Count  Percentage %
Country                             
United Kingdom  495478         91.43
Germany           9495          1.75
France            8557          1.58
EIRE              8196          1.51
Spain             2533          0.47
Netherlands       2371          0.44
Belgium           2069          0.38
Switzerland       2002          0.37
Portugal          1519          0.28
Australia         1259          0.23


## Step 7: Top Products by Frequency

Shows the most frequently appearing products in the transaction log.

In [16]:
df['Description'].value_counts().head(10)

Description
WHITE HANGING HEART T-LIGHT HOLDER    2369
REGENCY CAKESTAND 3 TIER              2200
JUMBO BAG RED RETROSPOT               2159
PARTY BUNTING                         1727
LUNCH BAG RED RETROSPOT               1638
ASSORTED COLOUR BIRD ORNAMENT         1501
SET OF 3 CAKE TINS PANTRY DESIGN      1473
PACK OF 72 RETROSPOT CAKE CASES       1385
LUNCH BAG  BLACK SKULL.               1350
NATURAL SLATE HEART CHALKBOARD        1280
Name: count, dtype: int64

## Step 8: Cancellation Rate

Invoices starting with 'C' are cancellations. Understanding the cancellation rate gives context for the cleaning decisions made in Notebook 02.

In [18]:
cancellations = df[df['InvoiceNo'].astype(str).str.startswith('C')]
total = len(df)
cancel_count = len(cancellations)
print(f"Total transactions:    {total:,}")
print(f"Cancellations:         {cancel_count:,}")
print(f"Cancellation rate:     {cancel_count/total*100:.1f}%")

Total transactions:    541,909
Cancellations:         9,288
Cancellation rate:     1.7%


## Step 9: Save Raw Backup

We save the raw data as a CSV backup before any cleaning. This allows us to reference the original dataset in later notebooks (e.g., for cancellation analysis).

In [20]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.to_csv("raw_data_backup.csv", index=False)
print("raw_data_backup.csv saved successfully.")

raw_data_backup.csv saved successfully.


## Summary

| Metric | Value |
|--------|-------|
| Total transactions | 541,909 |
| Columns | 8 |
| Missing CustomerIDs | ~135,080 (24.9%) |
| Cancellation rate | ~2% |
| Date range | Dec 2010 – Dec 2011 |
| Primary market | United Kingdom (~91%) |

**Next step:** Notebook 02 will clean the dataset by removing cancellations, missing CustomerIDs, invalid prices, and outliers.